# Task 3 (Ablation) — Hybrid Retrieval with Groundedness Verification, Few-Shot Prompting
Dataset: TriviaQA (`rc.nocontext`) — Project 10


## 0. Overview

This is a **separate, standalone experiment**, not the main Task 3 submission notebook. It reuses the exact same BM25+dense RRF fusion pipeline as `task3_hybrid_groundedness.ipynb`, and changes only the generator prompt: the same few-shot exemplars used in `task2_rag_fewshot.ipynb` (drawn from the TriviaQA `train` split, never used for training) are added to the chat history before the real question, for both the pass-1 and retry prompts. Writes to separate `task3_fewshot_*` result files so the zero-shot Task 3 results are never overwritten -- the two can be compared side by side (Section 10).

## 1. Setup

In [ ]:
# Importing the libraries we need for data handling, retrieval, and progress bars
import json
import re
import string
import random
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd
from tqdm.auto import tqdm


# Fixing the seed everywhere so retrieval and generation stay reproducible across runs
SEED = 42
random.seed(SEED)
np.random.seed(SEED)


# Detecting whether we're running on Colab (needed for the GPU generator) or locally
try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

print("Running on Colab:" if IN_COLAB else "Running locally:", IN_COLAB)

In [ ]:
# Installing Colab-only dependencies (skipped locally, since requirements.txt already covers them)
if IN_COLAB:
    get_ipython().system('pip install -q rank_bm25 sentence-transformers faiss-cpu nltk accelerate "bitsandbytes>=0.46.1"')
    print("Installed Colab-only deps. If bitsandbytes was just installed for the first time, "
          "Runtime > Restart session, then re-run from this cell.")
else:
    print("Local run: assuming `pip install -r ../requirements.txt` has already been run.")

In [ ]:
# Setting up the config constants used throughout this notebook
K_LIST = [1, 3, 5, 10]
TOP_K = 5
TOP_K_RETRIEVE = 10
RRF_K = 60  # spec: RRF constant = 60
MAX_PASSAGE_TOKENS = 200
MAX_NEW_TOKENS = 64
TEMPERATURE = 0.0
EMBED_MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"
GEN_MODEL_NAME = "Qwen/Qwen2.5-7B-Instruct"
GROUNDEDNESS_THRESHOLD = 0.5  # fraction of the answer's content words that must appear
                               # in the retrieved evidence to call it "grounded"


# Setting up the file names for the sample/index files and this notebook's own generation output
SAMPLE_FILENAME = "triviaqa_sample_300.jsonl"
INDEX_FILENAME = "wiki_paragraph_index.jsonl"
GENERATION_RESULTS_FILENAME = "task3_fewshot_generation_results.jsonl"


# On Colab, prompt to upload the two files Task 1 produced; locally, just point at the data/ folder
if IN_COLAB:
    from google.colab import files
    print(f"Upload {SAMPLE_FILENAME} and {INDEX_FILENAME} (from this repo's data/ folder):")
    uploaded = files.upload()
    assert SAMPLE_FILENAME in uploaded and INDEX_FILENAME in uploaded, (
        "Both files are required -- re-run this cell and upload them."
    )
    SAMPLE_PATH = Path(SAMPLE_FILENAME)
    INDEX_PATH = Path(INDEX_FILENAME)
    RESULTS_DIR = Path(".")
else:
    DATA_DIR = Path("../../23cs30011_project/data")
    RESULTS_DIR = Path("../../23cs30011_project/results")
    RESULTS_DIR.mkdir(parents=True, exist_ok=True)
    SAMPLE_PATH = DATA_DIR / SAMPLE_FILENAME
    INDEX_PATH = DATA_DIR / INDEX_FILENAME

GENERATION_RESULTS_PATH = RESULTS_DIR / GENERATION_RESULTS_FILENAME


# Loading the fixed 300-question sample and the paragraph index built in Task 1
samples = [json.loads(l) for l in open(SAMPLE_PATH)]
passages = [json.loads(l) for l in open(INDEX_PATH)]
print(f"Loaded {len(samples)} questions, {len(passages)} paragraphs (same as Tasks 1 & 2).")

In [ ]:
# Downloading the NLTK resources needed for tokenizing / POS-tagging / stopwords (skipped if already cached)
import nltk

for pkg in ["punkt", "punkt_tab", "averaged_perceptron_tagger", "averaged_perceptron_tagger_eng", "stopwords"]:
    try:
        nltk.data.find(f"tokenizers/{pkg}")
    except LookupError:
        try:
            nltk.download(pkg, quiet=True)
        except Exception:
            pass
    try:
        nltk.data.find(f"taggers/{pkg}")
    except LookupError:
        try:
            nltk.download(pkg, quiet=True)
        except Exception:
            pass
    try:
        nltk.data.find(f"corpora/{pkg}")
    except LookupError:
        try:
            nltk.download(pkg, quiet=True)
        except Exception:
            pass

from nltk.tokenize import sent_tokenize, word_tokenize
from nltk import pos_tag
from nltk.corpus import stopwords


# Loading English stopwords, extended with question words that would otherwise skew the groundedness overlap score
STOPWORDS = set(stopwords.words("english")) | {
    "who", "what", "when", "where", "which", "whom", "whose", "how", "many", "much"
}


# Setting up the tokenizer and the EM/F1/recall@k metric helpers (same as Tasks 1 & 2)
TOKEN_RE = re.compile(r"[a-z0-9]+")


def tokenize(text):
    """Utility function to lowercase and tokenize text for BM25 / overlap scoring."""
    return TOKEN_RE.findall(text.lower())


def normalize_answer(s):
    """SQuAD-style normalization: lowercase, remove punctuation, remove articles, fix whitespace."""
    s = s.lower()
    s = "".join(ch for ch in s if ch not in set(string.punctuation))
    s = re.sub(r"\b(a|an|the)\b", " ", s)
    s = " ".join(s.split())
    return s


def compute_em(pred, golds):
    """Utility function to return 1.0 if the prediction exactly matches any
    gold answer (after normalization), else 0.0."""
    norm_pred = normalize_answer(pred)
    return float(any(norm_pred == normalize_answer(g) for g in golds))


def f1_score(pred, gold):
    """Utility function that computes token-level F1 between one prediction
    and one gold answer string."""
    pred_tokens = normalize_answer(pred).split()
    gold_tokens = normalize_answer(gold).split()
    if len(pred_tokens) == 0 or len(gold_tokens) == 0:
        return float(pred_tokens == gold_tokens)
    common = Counter(pred_tokens) & Counter(gold_tokens)
    num_same = sum(common.values())
    if num_same == 0:
        return 0.0
    precision = num_same / len(pred_tokens)
    recall = num_same / len(gold_tokens)
    return 2 * precision * recall / (precision + recall)


def compute_f1(pred, golds):
    """Utility function that takes the best F1 across all acceptable gold answers."""
    return max(f1_score(pred, g) for g in golds) if golds else 0.0


def recall_at_k(hits, golds, k, passages_ref):
    """Utility function to check whether any gold answer string appears
    (as a normalized substring) in any of the top-k retrieved passages."""
    norm_golds = [normalize_answer(g) for g in golds]
    for pid, _ in hits[:k]:
        norm_text = normalize_answer(passages_ref[pid]["text"])
        if any(g and g in norm_text for g in norm_golds):
            return 1.0
    return 0.0


print("Shared helpers ready.")

## 2. Rebuild BM25 ranking (Task 1's method)

In [ ]:
# Rebuilding the same BM25 index from Task 1, over the same paragraph pool
from rank_bm25 import BM25Okapi

corpus_texts = [p["text"] for p in passages]
tokenized_corpus = [tokenize(t) for t in tqdm(corpus_texts, desc="Tokenizing corpus")]
bm25 = BM25Okapi(tokenized_corpus, k1=1.5, b=0.75)

def bm25_retrieve(question, top_k=TOP_K_RETRIEVE):
    """Utility function that scores every passage against the question with BM25
    and returns the top_k (passage_id, score) pairs, highest score first."""
    scores = bm25.get_scores(tokenize(question))
    top_idx = np.argsort(scores)[::-1][:top_k]
    return [(int(i), float(scores[i])) for i in top_idx]


# Iterating through every sampled question and retrieving its top-k BM25 hits
bm25_results = [bm25_retrieve(s["question"]) for s in tqdm(samples, desc="BM25 retrieval")]
print("BM25 index built over", len(tokenized_corpus), "paragraphs.")

## 3. Rebuild dense ranking (Task 2's method)

In [ ]:
# Rebuilding the same dense retriever from Task 2 (MiniLM embeddings + a FAISS inner-product index)
from sentence_transformers import SentenceTransformer
import faiss

embedder = SentenceTransformer(EMBED_MODEL_NAME)

passage_embeddings = embedder.encode(
    corpus_texts, batch_size=64, show_progress_bar=True, convert_to_numpy=True,
    normalize_embeddings=True,
).astype("float32")


# Building the FAISS index over the passage embeddings
dim = passage_embeddings.shape[1]
faiss_index = faiss.IndexFlatIP(dim)
faiss_index.add(passage_embeddings)


# Embedding every question the same way, then searching the index for its top-k passages
question_embeddings = embedder.encode(
    [s["question"] for s in samples], batch_size=64, show_progress_bar=True,
    convert_to_numpy=True, normalize_embeddings=True,
).astype("float32")

scores, indices = faiss_index.search(question_embeddings, TOP_K_RETRIEVE)
dense_results = [
    list(zip(indices[i].tolist(), scores[i].tolist())) for i in range(len(samples))
]
print("Dense index built over", faiss_index.ntotal, "paragraphs.")

## 4. Reciprocal Rank Fusion (constant=60)

In [ ]:
def rrf_fuse(list_a, list_b, k=RRF_K, top_k=TOP_K_RETRIEVE):
    """Utility function that fuses two ranked lists with Reciprocal Rank Fusion:
    each passage's score is the sum of 1/(k+rank) across both lists."""
    scores = {}
    for rank, (pid, _) in enumerate(list_a, start=1):
        scores[pid] = scores.get(pid, 0.0) + 1.0 / (k + rank)
    for rank, (pid, _) in enumerate(list_b, start=1):
        scores[pid] = scores.get(pid, 0.0) + 1.0 / (k + rank)
    ranked = sorted(scores.items(), key=lambda x: x[1], reverse=True)[:top_k]
    return ranked



# Fusing BM25 and dense rankings for every question with Reciprocal Rank Fusion
hybrid_results = [rrf_fuse(bm25_results[i], dense_results[i]) for i in range(len(samples))]

print("Example hybrid retrieval for:", samples[0]["question"])
for pid, sc in hybrid_results[0][:3]:
    print(f"  rrf={sc:.4f} doc={passages[pid]['doc_title']!r} text={passages[pid]['text'][:100]!r}")

## 5. recall@k: BM25 vs Dense vs Hybrid

In [ ]:
def recall_table(results):
    """Utility function that averages recall@k (for every k in K_LIST) over all
    questions, given one set of retrieval results."""
    rows = []
    for s, hits in zip(samples, results):
        rows.append({f"recall@{k}": recall_at_k(hits, s["gold_answers"], k, passages) for k in K_LIST})
    df = pd.DataFrame(rows)
    return {f"recall@{k}": df[f"recall@{k}"].mean() for k in K_LIST}



# Comparing recall@k across BM25-only, dense-only, and the hybrid fusion
retrieval_comparison = pd.DataFrame({
    "BM25 (Task 1)": recall_table(bm25_results),
    "Dense (Task 2)": recall_table(dense_results),
    "Hybrid RRF (Task 3)": recall_table(hybrid_results),
})
print(retrieval_comparison)

## 5b. Pick few-shot exemplars from the TriviaQA `train` split

Same fixed exemplar-selection procedure as `task2_rag_fewshot.ipynb` (seed 42, short canonical answers only) -- in-context demonstrations, never used for training.

In [ ]:
# Picking a fixed set of few-shot exemplars from TriviaQA's train split -- never used for
# training, only prepended to the generator's prompt as in-context demonstrations
from datasets import load_dataset


# Few-shot config: how many exemplars to use, and the max length of an answer to qualify as a clean one
N_FEWSHOT = 3
MAX_FEWSHOT_ANSWER_CHARS = 20


# Loading the TriviaQA train split just to draw demonstration examples from (not for training)
train_ds = load_dataset("mandarjoshi/trivia_qa", "rc.nocontext", split="train")


# Shuffling train indices deterministically so the exemplars picked are reproducible
rng_fewshot = random.Random(SEED)
train_indices = list(range(len(train_ds)))
rng_fewshot.shuffle(train_indices)


# Iterating through the shuffled train rows, keeping short-answer ones until we have N_FEWSHOT exemplars
fewshot_examples = []
for idx in train_indices:
    row = train_ds[idx]
    ans = row["answer"]["value"]
    if ans and len(ans) <= MAX_FEWSHOT_ANSWER_CHARS:
        fewshot_examples.append({"question": row["question"], "answer": ans})
    if len(fewshot_examples) >= N_FEWSHOT:
        break

print(f"Picked {len(fewshot_examples)} few-shot exemplars from train (seed={SEED}, never used for training):")
for ex in fewshot_examples:
    print(" -", ex)

## 6. Generation: hybrid top-5 -> `Qwen2.5-7B-Instruct` (4-bit), greedy decode; retry ungrounded with top-10

In [ ]:
def groundedness_overlap(pred, hits, top_k):
    """Utility function that scores how much of the prediction's content-word
    vocabulary is actually present in the top-k retrieved evidence."""
    pred_tokens = set(tokenize(pred)) - STOPWORDS
    if not pred_tokens:
        return 0.0
    evidence_text = " ".join(passages[pid]["text"] for pid, _ in hits[:top_k])
    evidence_tokens = set(tokenize(evidence_text)) - STOPWORDS
    return len(pred_tokens & evidence_tokens) / len(pred_tokens)




# Only load the generator model when running on Colab (needs a GPU); locally we just
# reuse a previous run's saved predictions instead
if IN_COLAB:
    import time
    import torch
    from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

    torch.manual_seed(SEED)

    
    # Loading Qwen2.5-7B-Instruct in 4-bit so it fits on a single Colab GPU
    quant_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_compute_dtype=torch.float16,
        bnb_4bit_quant_type="nf4",
    )

    print("Loading tokenizer/model:", GEN_MODEL_NAME)
    gen_tokenizer = AutoTokenizer.from_pretrained(GEN_MODEL_NAME)
    gen_model = AutoModelForCausalLM.from_pretrained(
        GEN_MODEL_NAME, quantization_config=quant_config, device_map="auto",
    )
    gen_model.eval()

    
    # Utility function to truncate a passage to a fixed token budget before it goes in the prompt
    def truncate_to_tokens(text, max_tokens):
        ids = gen_tokenizer.encode(text, add_special_tokens=False)
        if len(ids) <= max_tokens:
            return text
        return gen_tokenizer.decode(ids[:max_tokens], skip_special_tokens=True)

    
    # Building the generator's prompt: the retrieved passages, the question, and (on retry)
    # a note that the first answer was not grounded, asking it to look again at a wider context
    def build_prompt(question, top_passages, top_k, retry=False):
        passage_block = "\n\n".join(
            f"Passage {i+1} ({passages[pid]['doc_title']}): "
            f"{truncate_to_tokens(passages[pid]['text'], MAX_PASSAGE_TOKENS)}"
            for i, (pid, _) in enumerate(top_passages[:top_k])
        )
        if retry:
            user_msg = (
                "Your previous answer did not clearly match the retrieved evidence. Here is a "
                "WIDER set of passages. Re-read them carefully. Answer using ONLY information "
                "explicitly stated in the passages below. Respond with just the short answer, "
                "no explanation. If the answer is genuinely not present in these passages, "
                "respond exactly: UNKNOWN\n\n"
                f"{passage_block}\n\nQuestion: {question}\nAnswer:"
            )
        else:
            user_msg = (
                "Answer the question using ONLY the information in the passages below. "
                "Respond with just the short answer, no explanation, no full sentence.\n\n"
                f"{passage_block}\n\nQuestion: {question}\nAnswer:"
            )
        # Few-shot: prepend (question, answer) exemplars from `train` as prior chat turns,
        # purely in-context -- no weights are touched here or anywhere in this project.
        messages = [
            {"role": "system", "content": "You are a concise, factual question-answering assistant."},
        ]
        for ex in fewshot_examples:
            messages.append({"role": "user", "content": f"Question: {ex['question']}\nAnswer:"})
            messages.append({"role": "assistant", "content": ex["answer"]})
        messages.append({"role": "user", "content": user_msg})
        return gen_tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

    def generate_answer(question, top_passages, top_k, retry=False):
        """Utility function that renders the prompt, runs greedy generation, and decodes
        just the newly generated tokens."""
        prompt = build_prompt(question, top_passages, top_k, retry=retry)
        inputs = gen_tokenizer(prompt, return_tensors="pt").to(gen_model.device)
        with torch.no_grad():
            out = gen_model.generate(
                **inputs, max_new_tokens=MAX_NEW_TOKENS, do_sample=False,
                pad_token_id=gen_tokenizer.eos_token_id,
            )
        new_tokens = out[0][inputs["input_ids"].shape[1]:]
        return gen_tokenizer.decode(new_tokens, skip_special_tokens=True).strip()

    # Pass 1: generate with top-5, same settings as Task 2.
    generations = []
    t0 = time.time()
    for i, (s, hits) in enumerate(zip(samples, hybrid_results)):
        pred = generate_answer(s["question"], hits, top_k=TOP_K)
        overlap = groundedness_overlap(pred, hits, top_k=TOP_K)
        generations.append({
            "question_id": s["question_id"], "question": s["question"],
            "gold_answers": s["gold_answers"], "prediction_raw": pred,
            "overlap": overlap, "is_grounded": overlap >= GROUNDEDNESS_THRESHOLD,
        })
        if (i + 1) % 10 == 0:
            elapsed = time.time() - t0
            print(f"Pass 1: {i+1}/{len(samples)} done, {elapsed:.0f}s elapsed")

    n_ungrounded = sum(1 for g in generations if not g["is_grounded"])
    print(f"\nPass 1 done. {n_ungrounded}/{len(samples)} flagged ungrounded -> retrying with top-10.")

    # Pass 2: retry only the flagged-ungrounded questions with expanded top-10 context.
    t0 = time.time()
    retried = 0
    for i, (s, hits, g) in enumerate(zip(samples, hybrid_results, generations)):
        if g["is_grounded"]:
            g["prediction_retry"] = g["prediction_raw"]  # unchanged
            continue
        retry_pred = generate_answer(s["question"], hits, top_k=TOP_K_RETRIEVE, retry=True)
        g["prediction_retry"] = retry_pred
        retried += 1
        if retried % 10 == 0:
            elapsed = time.time() - t0
            print(f"Pass 2 (retry): {retried}/{n_ungrounded} done, {elapsed:.0f}s elapsed")

    print(f"\nPass 2 done. {retried} questions retried.")

    
    # Persisting pass-1 and pass-2 (retry) predictions so this does not need to be rerun on Colab again
    with open(GENERATION_RESULTS_PATH, "w") as f:
        for r in generations:
            f.write(json.dumps(r) + "\n")
    print("Saved", GENERATION_RESULTS_PATH)
    print("\nDownload this file and place it in this repo's results/ folder:")
    files.download(str(GENERATION_RESULTS_PATH))


# Local run: skip generation entirely and just load predictions from a previous Colab run
else:
    assert GENERATION_RESULTS_PATH.exists(), (
        f"{GENERATION_RESULTS_PATH} not found. Run this notebook on Colab (GPU) first to "
        "generate it, or place a previous run's output there."
    )
    generations = [json.loads(l) for l in open(GENERATION_RESULTS_PATH)]
    print(f"Local run: loaded {len(generations)} pre-generated predictions from {GENERATION_RESULTS_PATH}.")


# Sanity check: the loaded/generated predictions must cover exactly the fixed 300-question sample
gen_by_qid = {g["question_id"]: g for g in generations}
assert set(gen_by_qid) == {s["question_id"] for s in samples}, "generation set doesn't match the sample set"
print("Question IDs match the fixed sample exactly.")

## 7. Score hybrid generations (pre-groundedness-fallback): EM, F1

In [ ]:
# Iterating through every question and building one results record per question,
# scored against the pass-1 (no retry) prediction
records = []
for s, hits in zip(samples, hybrid_results):
    g = gen_by_qid[s["question_id"]]
    pred = g["prediction_raw"]
    golds = s["gold_answers"]
    records.append({
        "question_id": s["question_id"],
        "question": s["question"],
        "gold_answers": golds,
        "prediction_raw": pred,
        "prediction_retry": g["prediction_retry"],
        "overlap": g["overlap"],
        "is_grounded": g["is_grounded"],
        "em_raw": compute_em(pred, golds),
        "f1_raw": compute_f1(pred, golds),
        "top1_doc": passages[hits[0][0]]["doc_title"],
        "hybrid_hits": hits,
    })

task3_fewshot_df = pd.DataFrame(records)


# Averaging EM/F1 over all 300 questions; recall@k is just carried over from the hybrid retrieval table
raw_metrics = {"EM": task3_fewshot_df["em_raw"].mean(), "F1": task3_fewshot_df["f1_raw"].mean()}
for k in K_LIST:
    raw_metrics[f"recall@{k}"] = retrieval_comparison["Hybrid RRF (Task 3)"][f"recall@{k}"]


# Loading the other tasks' metrics (if present) so we can print them all side by side for comparison
task1_metrics_path = Path("../../results/task1_metrics.json")
task2_metrics_path = Path("../../results/task2_metrics.json")
task3_zeroshot_metrics_path = Path("../../results/task3_metrics.json")
compare_cols = {"Task 3 FEW-SHOT hybrid (HEADLINE, pass-1)": raw_metrics}
if task1_metrics_path.exists():
    compare_cols["Task 1 (BM25 + heuristic)"] = json.load(open(task1_metrics_path))
if task2_metrics_path.exists():
    compare_cols["Task 2 zero-shot (dense + LLM)"] = json.load(open(task2_metrics_path))
if task3_zeroshot_metrics_path.exists():
    compare_cols["Task 3 zero-shot (HEADLINE, pass-1)"] = json.load(open(task3_zeroshot_metrics_path))["pass1_HEADLINE"]

print(pd.DataFrame(compare_cols))

## 8. Groundedness check: lexical overlap between answer and retrieved evidence

In [ ]:
# Checking whether the groundedness-overlap heuristic actually tracks correctness:
# do correct answers tend to come back grounded, and incorrect ones ungrounded?
correct = task3_fewshot_df["em_raw"] == 1.0
incorrect = ~correct
frac_correct_grounded = task3_fewshot_df.loc[correct, "is_grounded"].mean()
frac_incorrect_ungrounded = (~task3_fewshot_df.loc[incorrect, "is_grounded"]).mean()

print(f"Correct answers with high overlap (grounded):   {frac_correct_grounded:.3f} "
      f"({correct.sum()} correct total)")
print(f"Incorrect answers with low overlap (ungrounded): {frac_incorrect_ungrounded:.3f} "
      f"({incorrect.sum()} incorrect total)")
print(f"Groundedness-check accuracy "
      f"(is_grounded predicts correctness): "
      f"{(task3_fewshot_df['is_grounded'] == correct).mean():.3f}")

## 9. Experiment: does retrying ungrounded answers (top-10 + "say UNKNOWN") help?

In [ ]:
# Iterating through every question and picking the final prediction: the pass-1 answer
# if it was grounded, otherwise the retried (top-10) answer
final_predictions = []
retry_used = []
for row in task3_fewshot_df.itertuples():
    if row.is_grounded:
        final_predictions.append(row.prediction_raw)
        retry_used.append(False)
    else:
        final_predictions.append(row.prediction_retry)
        retry_used.append(True)


# Recording the final predictions back onto the dataframe and recomputing EM/F1 against them
task3_fewshot_df["prediction_final"] = final_predictions
task3_fewshot_df["retry_used"] = retry_used
task3_fewshot_df["em_final"] = [compute_em(p, g) for p, g in zip(final_predictions, task3_fewshot_df["gold_answers"])]
task3_fewshot_df["f1_final"] = [compute_f1(p, g) for p, g in zip(final_predictions, task3_fewshot_df["gold_answers"])]

print(f"Retry used on {sum(retry_used)}/{len(task3_fewshot_df)} questions "
      f"({sum(retry_used)/len(task3_fewshot_df):.1%}).")

## 10. Final comparison: Task 1 vs Task 2 vs Task 3, zero-shot vs few-shot (headline = pass-1; retry shown for reference)

In [ ]:
# Saving the per-question results, plus the headline (pass-1), reference (with-retry),
# and groundedness metrics
task3_fewshot_df.to_csv(RESULTS_DIR / "task3_fewshot_per_question_results.csv", index=False)

final_metrics = {
    "n_questions": len(task3_fewshot_df),
    "EM": task3_fewshot_df["em_final"].mean(),
    "F1": task3_fewshot_df["f1_final"].mean(),
}
for k in K_LIST:
    final_metrics[f"recall@{k}"] = retrieval_comparison["Hybrid RRF (Task 3)"][f"recall@{k}"]

groundedness_metrics = {
    "frac_correct_grounded": float(frac_correct_grounded),
    "frac_incorrect_ungrounded": float(frac_incorrect_ungrounded),
    "groundedness_check_accuracy": float((task3_fewshot_df["is_grounded"] == correct).mean()),
    "retry_rate": float(sum(retry_used) / len(task3_fewshot_df)),
}

with open(RESULTS_DIR / "task3_fewshot_metrics.json", "w") as f:
    json.dump({"pass1_HEADLINE": raw_metrics, "with_retry_reference_only": final_metrics,
               "groundedness": groundedness_metrics}, f, indent=2)


# Adding the with-retry numbers to the comparison table just for reference -- the headline
# result stays pass-1
compare_cols["Task 3 few-shot retry experiment (reference)"] = final_metrics
print(pd.DataFrame(compare_cols))
print("\nGroundedness:", json.dumps(groundedness_metrics, indent=2))
print(f"\nRetry net effect: {sum((task3_fewshot_df['retry_used']) & (task3_fewshot_df['em_raw']==0) & (task3_fewshot_df['em_final']==1))} fixed, "
      f"{sum((task3_fewshot_df['retry_used']) & (task3_fewshot_df['em_raw']==1) & (task3_fewshot_df['em_final']==0))} broken "
      f"-> headline Task 3 result uses PASS-1 (no retry), not this column.")


# On Colab, offer the two result files for download so they can be placed in this repo's results/ folder
if IN_COLAB:
    files.download(str(RESULTS_DIR / "task3_fewshot_per_question_results.csv"))
    files.download(str(RESULTS_DIR / "task3_fewshot_metrics.json"))

## 11. Qualitative analysis: retry corrections and misses

In [ ]:
def show_retry_examples(df, n=5, outcome="fixed", seed=SEED):
    """Utility function to print n random retry examples of the given outcome
    ('fixed', 'broke', or anything else for 'still wrong'), for qualitative inspection."""
    fb = df[df["retry_used"]]
    if outcome == "fixed":
        subset = fb[(fb["em_raw"] == 0) & (fb["em_final"] == 1)]
    elif outcome == "broke":
        subset = fb[(fb["em_raw"] == 1) & (fb["em_final"] == 0)]
    else:
        subset = fb[(fb["em_raw"] == 0) & (fb["em_final"] == 0)]
    subset = subset.sample(min(n, len(subset)), random_state=seed)
    for row in subset.itertuples():
        print(f"Q: {row.question}")
        print(f"   Gold: {row.gold_answers}")
        print(f"   Pass-1 (top-5, overlap={row.overlap:.2f}): {row.prediction_raw!r}")
        print(f"   Retry (top-10): {row.prediction_retry!r}")
        print("-" * 80)



# Printing a few real examples of retry fixing an error, and a few where retry broke
# a previously correct answer
print(f">>> Retry FIXED an error ({((task3_fewshot_df['retry_used']) & (task3_fewshot_df['em_raw']==0) & (task3_fewshot_df['em_final']==1)).sum()} cases)")
show_retry_examples(task3_fewshot_df, outcome="fixed")

print(f"\n>>> Retry BROKE a correct answer ({((task3_fewshot_df['retry_used']) & (task3_fewshot_df['em_raw']==1) & (task3_fewshot_df['em_final']==0)).sum()} cases)")
show_retry_examples(task3_fewshot_df, outcome="broke")

## 12. Summary

Run this notebook on Colab to fill in the headline (pass-1) EM/F1/recall@k above, then compare against `results/task3_metrics.json` (zero-shot) to decide whether few-shot replaces the official Task 3 submission notebook.